In [1]:
OPENAI_KEY = "INSERT_KEY"
GEMINI_KEY = "INSERT_KEY"
CLAUDE_KEY = "INSERT_KEY"

In [3]:
import os, re, time
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
import matplotlib.pyplot as plt
import google.generativeai as genai
from sentence_transformers import SentenceTransformer

# --- 2. UTILITY FUNCTIONS ---
def super_clean_id(filename):
    nums = re.findall(r"(\d+)", str(filename))
    return str(int(nums[-1])) if nums else str(filename).split('.')[0].strip().lower()

def load_dir_robust(folder):
    texts = {}
    if not folder.is_dir(): return texts
    for fn in os.listdir(folder):
        if fn.lower().endswith(".txt"):
            eid = super_clean_id(fn)
            with open(folder / fn, "r", encoding="utf-8", errors="ignore") as f:
                texts[eid] = f.read().strip()
    return texts

def get_embeddings_batched(text_list, model_id):
    if "models/" in model_id:
        result = genai.embed_content(model=model_id, content=text_list, task_type="clustering")
        return np.array(result['embedding'])
    return SentenceTransformer(model_id).encode(text_list, show_progress_bar=False)


# --- 1. CONFIGURATION ---
genai.configure(api_key=GEMINI_KEY)

ARGREWRITE_ROOT = Path("../ArgRewrite").resolve()
D1_PATH = ARGREWRITE_ROOT / "dataset" / "essays" / "Draft1"
D2_PATH = ARGREWRITE_ROOT / "dataset" / "essays" / "Draft2"

EMBEDDING_MODELS = {
    "MiniLM-L6": "all-miniLM-L6-v2",
    "Gemini-004": "models/text-embedding-004"
}
# CUSTOMIZE LLM NAMES HERE
# Format: "FOLDER_NAME_UPPER": "Display Name"

# --- 1. CONFIGURATION ---
LLM_DISPLAY_NAMES = {
    "CLAUDE-HAIKU-4-5-20251001": "claude 3.5 haiku",
    "GEMINI-2.5-FLASH": "gemini 2.5 flash",
    "GPT-5-MINI": "gpt-5-mini",
    # "GPT-4O-MINI": "gpt-4o-mini"
}

VARIANT_COLORS = {
    "minimal": "#d95f02", "grammar": "#1b9e77", "general": "#b2182b",
    "completion": "#ce1256", "expansion": "#7570b3"
}

DESCRIPTIVE_NAMES = {
    "minimal": "Minimal Edit", "grammar": "Grammar Edits",
    "completion": "Completion", "expansion": "Expansion", "general": "General Revision",
}

PLOT_ORDER = ["general", "minimal", "grammar", "completion", "expansion"]

# --- 3. DATA AGGREGATION ---
d1_map, d2_map = load_dir_robust(D1_PATH), load_dir_robust(D2_PATH)
common_ids = sorted(set(d1_map.keys()) & set(d2_map.keys()), key=lambda x: int(x))
df_human = pd.DataFrame([{"essay_id": eid, "draft1": d1_map[eid], "draft2": d2_map[eid]} for eid in common_ids])

MODEL_DIRS = [p for p in Path(".").iterdir() if p.is_dir() and p.name.startswith("llm_drafts")]
all_data_list = []

for folder in MODEL_DIRS:
    model_tag = folder.name.replace("llm_drafts_", "").upper()
    for csv_path in folder.glob("*.csv"):
        v_key = "general" if any(x in csv_path.name.lower() for x in ["expert", "general"]) \
                else next((v for v in PLOT_ORDER if v in csv_path.name.lower()), None)
        if not v_key: continue
        ldf = pd.read_csv(csv_path)
        ldf['essay_id'] = ldf['essay_id'].apply(super_clean_id)
        ai_col = 'output_text' if 'output_text' in ldf.columns else 'final_llm_text'
        merged = pd.merge(df_human, ldf[['essay_id', ai_col]], on='essay_id').rename(columns={ai_col: 'ai_text'})
        merged['llm_model'], merged['mode'] = model_tag, v_key
        merged['feedback'] = "WITH" if "with_FB" in csv_path.name else "WITHOUT"
        all_data_list.append(merged)

df_master = pd.concat(all_data_list)

/tmp/ipykernel_423484/2249286789.py:8: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


NameError: name 'GEMINI_KEY' is not defined

In [ ]:
import os, re
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# --- 1. CONFIGURATION & COLOR PALETTE ---
NRC_LEXICON_PATH = Path("../NRC-Emotion-Lexicon/NRC-Emotion-Lexicon-Wordlevel-v0.92.txt")
ARGREWRITE_ROOT = Path("../ArgRewrite").resolve()
D1_PATH = ARGREWRITE_ROOT / "dataset" / "essays" / "Draft1"
D2_PATH = ARGREWRITE_ROOT / "dataset" / "essays" / "Draft2"

COLOR_MAP = {
    "Human (D2-D1)": "#777777",
    "CLAUDE-HAIKU-4-5-20251001": "#d73027",
    "GEMINI-2.5-FLASH": "#4575b4",
    "GPT-5-MINI": "#1b9e77"
}

LLM_DISPLAY_NAMES = {
    "CLAUDE-HAIKU-4-5-20251001": "claude-3.5-haiku",
    "GEMINI-2.5-FLASH": "gemini-2.5-flash",
    "GPT-5-MINI": "gpt-5-mini"
}

EMOTIONS = ["anger", "anticipation", "disgust", "fear", "joy", "sadness", "surprise", "trust", "positive", "negative"]
PLOT_ORDER = ["general", "minimal", "grammar", "completion", "expansion"]

# --- 2. HELPER FUNCTIONS ---

def load_nrc_lexicon(path: Path):
    if not path.exists():
        print(f"Error: NRC Lexicon not found at {path}")
        return {}
    lex = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) == 3 and int(parts[2]) == 1:
                lex.setdefault(parts[0].lower(), {})[parts[1].lower()] = 1
    return lex

NRC = load_nrc_lexicon(NRC_LEXICON_PATH)

def get_emotion_vector(text):
    tokens = re.findall(r"[A-Za-z']+", str(text).lower())
    n_words = max(1, len(tokens))
    counts = {e: 0 for e in EMOTIONS}
    for w in tokens:
        if w in NRC:
            for e in EMOTIONS: 
                counts[e] += NRC[w].get(e, 0)
    return np.array([(counts[e] / n_words) * 100.0 for e in EMOTIONS])

def super_clean_id(filename):
    nums = re.findall(r"(\d+)", str(filename))
    return str(int(nums[-1])) if nums else str(filename).split('.')[0].strip().lower()

def load_dir_texts(folder: Path):
    texts = {}
    if folder.is_dir():
        for fn in os.listdir(folder):
            if fn.lower().endswith(".txt"):
                eid = super_clean_id(fn)
                with open(folder / fn, "r", encoding="utf-8", errors="ignore") as f:
                    texts[eid] = f.read().strip()
    return texts

# --- 3. PLOTTING ENGINE ---

def plot_combined_emotion_bars(mode, feedback_status, data_subset, df_h):
    models = sorted(data_subset['llm_model'].unique())
    
    # Calculate Human Vectors
    h_d1 = np.mean([get_emotion_vector(t) for t in df_h['draft1']], axis=0)
    h_d2 = np.mean([get_emotion_vector(t) for t in df_h['draft2']], axis=0)
    
    # --- NEW INTERPRETATION: % Change relative to D1 ---
    # Formula: ((D2 - D1) / D1) * 100
    # We use np.where to handle cases where D1 might be 0
    h_rel_change = np.where(h_d1 > 0, ((h_d2 - h_d1) / h_d1) * 100, 0)

    plot_dict = {"Human (D2-D1)": h_rel_change} 
    
    for m_tag in models:
        m_df = data_subset[data_subset['llm_model'] == m_tag]
        ai_vec = np.mean([get_emotion_vector(t) for t in m_df['ai_text']], axis=0)
        d1_vec = np.mean([get_emotion_vector(t) for t in m_df['draft1']], axis=0)
        
        # Calculate Relative Change for LLM
        m_rel_change = np.where(d1_vec > 0, ((ai_vec - d1_vec) / d1_vec) * 100, 0)
        plot_dict[m_tag] = m_rel_change

    fig, ax = plt.subplots(figsize=(13, 7))
    x_indices = np.arange(len(EMOTIONS))
    n_groups = len(plot_dict)
    bar_width = 0.8 / n_groups

    for i, (key, values) in enumerate(plot_dict.items()):
        color = COLOR_MAP.get(key, "#999999") 
        label = LLM_DISPLAY_NAMES.get(key, key)
        offset = (i - (n_groups - 1) / 2) * bar_width
        ax.bar(x_indices + offset, values, width=bar_width, label=label, 
               color=color, edgecolor='white', linewidth=0.5)

    ax.legend(title="Editor", loc='upper left', frameon=True, framealpha=0.9)

    clean_mode = mode.replace("_", " ").title() + " Revision"
    clean_fb = "With Feedback" if feedback_status == "WITH_FB" else "Without Feedback"
    
    ax.set_title(f"Emotional Shift: {clean_mode} ({clean_fb})", 
                 fontsize=16, pad=25, fontweight='bold')

    ax.axhline(0, color='black', linewidth=1)
    
    # Updated Y-Label for better interpretation
    ax.set_ylabel("% Increase / Decrease relative to Original Draft", fontsize=11)
    ax.set_xlabel("NRC Emotion Categories", fontsize=11, labelpad=10)

    ax.set_xticks(x_indices)
    ax.set_xticklabels([e.capitalize() for e in EMOTIONS], fontsize=10)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    
    # Optional: Add a % sign to y-axis ticks
    from matplotlib.ticker import PercentFormatter
    ax.yaxis.set_major_formatter(PercentFormatter())
    
    plt.tight_layout()


# --- 4. DATA LOADING & EXECUTION (REMOVED DUPLICATES) ---

# 1. Load Human Baselines
print("Loading human essays...")
d1_map, d2_map = load_dir_texts(D1_PATH), load_dir_texts(D2_PATH)
common_ids = sorted(set(d1_map.keys()) & set(d2_map.keys()), key=lambda x: int(x))
df_human = pd.DataFrame([{"essay_id": eid, "draft1": d1_map[eid], "draft2": d2_map[eid]} for eid in common_ids])

# 2. Load LLM CSVs
print("Loading LLM drafts...")
all_data_list = []
MODEL_DIRS = [p for p in Path(".").iterdir() if p.is_dir() and p.name.startswith("llm_drafts")]

for folder in MODEL_DIRS:
    model_tag = folder.name.replace("llm_drafts_", "").upper()
    if model_tag not in LLM_DISPLAY_NAMES: continue
    
    for csv_path in folder.glob("*.csv"):
        v_key = "general" if any(x in csv_path.name.lower() for x in ["expert", "general"]) \
                else next((v for v in PLOT_ORDER if v in csv_path.name.lower()), None)
        if not v_key: continue
        
        ldf = pd.read_csv(csv_path)
        ldf['essay_id'] = ldf['essay_id'].apply(super_clean_id)
        ai_col = 'output_text' if 'output_text' in ldf.columns else 'final_llm_text'
        
        merged = pd.merge(df_human, ldf[['essay_id', ai_col]], on='essay_id').rename(columns={ai_col: 'ai_text'})
        merged['llm_model'], merged['mode'] = model_tag, v_key
        merged['feedback'] = "WITH_FB" if "with_FB" in csv_path.name else "NO_FB"
        all_data_list.append(merged)

if all_data_list:
    df_master = pd.concat(all_data_list)
    
    # 3. Generate the Plots (Running exactly once)
    for mode in PLOT_ORDER:
        for fb in ["NO_FB", "WITH_FB"]:
            subset = df_master[(df_master['mode'] == mode) & (df_master['feedback'] == fb)]
            if not subset.empty:
                plot_combined_emotion_bars(mode, fb, subset, df_human)
                plt.show()
                plt.close()